# Importar Librerías

In [2]:
import pandas as pd
import os

# 2 Pipeline  de Limpieza

## 2.1 Carga de datos

In [3]:
df = pd.read_csv("../data/raw/events.csv")
df = df.dropna(subset=["user_session"]) # Eliminar filas sin user_session

In [4]:
df

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2020-09-24 11:57:06 UTC,view,1996170,2144415922528452715,electronics.telephone,NaN,31.90,1515915625519388267,LJuJVLEjPT
1,2020-09-24 11:57:26 UTC,view,139905,2144415926932472027,computers.components.cooler,zalman,17.16,1515915625519380411,tdicluNnRY
2,2020-09-24 11:57:27 UTC,view,215454,2144415927158964449,NaN,NaN,9.81,1515915625513238515,4TMArHtXQy
3,2020-09-24 11:57:33 UTC,view,635807,2144415923107266682,computers.peripherals.printer,pantum,113.81,1515915625519014356,aGFYrNgC08
4,2020-09-24 11:57:36 UTC,view,3658723,2144415921169498184,NaN,cameronsino,15.87,1515915625510743344,aa4mmk0kwQ
...,...,...,...,...,...,...,...,...,...
885124,2021-02-28 23:55:01 UTC,view,953226,2144415927553229037,NaN,NaN,219.94,1515915625611023730,FRLqIttxKU
885125,2021-02-28 23:58:05 UTC,view,1715907,2144415927049912542,electronics.video.tv,starwind,80.03,1515915625611024014,g6WqPf50Ma
885126,2021-02-28 23:58:09 UTC,view,4170534,2144415939364389423,electronics.clocks,amazfit,64.92,1515915625611024020,xNIJBqZdkd
885127,2021-02-28 23:58:14 UTC,view,888273,2144415921932861531,electronics.telephone,NaN,10.16,1515915625611024030,9pCbKMIcSx


## 2.3 INFERENCIA DE CATEGORÍAS — productos sin category_code
=======================================================
Estrategia: mapear category_id → categoría inferida basándose en las marcas dominantes de cada grupo. Los que no se pueden inferir → "accessories.generic"

In [8]:
# MAPA DE INFERENCIA: category_id → categoria inferida
# Construido analizando las marcas dominantes de cada grupo

mapa_categorias = {
    # Accesorios / baterías de celular (cameronsino, samsung, xiaomi, nokia)
    2144415921169498184: "electronics.telephone.accessory",

    # Refrigeración PC (aerocool, thermaltake, zalman, formula)
    2144415926848585945: "computers.components.cooling",

    # Cables y adaptadores (greenconnect, orient, espada, cablexpert)
    2144415927158964449: "computers.accessories.cables",

    # Memorias RAM (kingston, hyperx, corsair, ballistix, crucial)
    2144415923535085701: "computers.components.memory",

    # Audio/video doméstico (panasonic, sony, hyundai, lg, supra)
    2144415926219440328: "electronics.audio_video.home",

    # Accesorios móviles (palmexx, greenconnect, smartbuy, hoco, baseus)
    2144415922201296994: "electronics.mobile.accessories",

    # Audio portátil (blast, panasonic, ritmix, sony)
    2144415926370435276: "electronics.audio.portable",

    # Memorias flash / tarjetas SD (transcend, sandisk, samsung, kingston)
    2144415921001726020: "computers.storage.flash",

    # Pequeños electrodomésticos (redmond)
    2144415928811520270: "appliances.small.kitchen",

    # Videojuegos / consolas (microsoft, sega, sony, palmexx, dendy)
    2144415922050302046: "electronics.gaming",

    # Afeitadoras / cuidado personal (panasonic, braun, philips)
    2144415922939494519: "appliances.personal_care",

    # Cables de red (greenconnect, orient, palmexx, telecom)
    2144415922234851427: "computers.network.cables",

    # Accesorios para tablets/phones (palmexx, samsung, baseus, wiwu, hp)
    2144415921815421016: "electronics.tablet.accessories",

    # Pequeños electrodomésticos cocina (kitfort, scarlett, polaris)
    2144415942703055498: "appliances.kitchen.small",

    # Herramientas de reparación (mechanic, cyberflux, solins, kester)
    2144415935673401802: "construction.tools.repair",

    # Accesorios / baterías varios (samsung, pitatel, buro, palmexx)
    2144415921505042512: "electronics.accessories.generic",

    # Herramientas eléctricas (hammer, bort, makita, bosch)
    2144415942350733951: "construction.tools.electric",

    # Electrodomésticos grandes cocina (gorenje, kitfort, bosch, philips)
    2144415925581906105: "appliances.kitchen.large",

    # Soportes y montaje (kromax, hama, onkron, ultramounts)
    2144415928333369601: "electronics.mounts",

    # Baterías herramientas (pitatel, topon, makita, dewalt)
    2144415942904382094: "construction.tools.batteries",

    # Eléctrica / UPS (iek, ippon, exegate, ekf, sven)
    2144415938206761488: "electronics.power.ups",

    # Herramientas de jardín (bort, karcher, patriot, huter, bosch)
    2144415927553229037: "construction.tools.garden",

    # Audio para auto (pioneer, swat, ural, aura, jbl)
    2144415933794353554: "auto.audio",

    # Pequeños electrodomésticos varios (panasonic, redmond, gorenje)
    2144415924256506007: "appliances.small.generic",

    # SAI / UPS (ippon, powercom, cyberpower, apc)
    2144415923610583175: "electronics.power.sai",

    # Generadores / compresores (fubag, berkut, hyundai, ritmix)
    2144415944036844204: "construction.tools.generators",

    # Memorias USB / almacenamiento (transcend, smartbuy, kingston)
    2144415921337270348: "computers.storage.usb",

    # Adaptadores / hubs (espada, palmexx, alas, wispen)
    2144415921253384266: "computers.accessories.hubs",

    # Herramientas de soldadura / reparación (codyson, element, kaisi)
    2144415935698567627: "construction.tools.soldering",

    # Accesorios almacenamiento (procase, palmexx, agestar, orient)
    2144415927645503728: "computers.storage.accessories",

    # Cables de red profesional (greenconnect, mikrotik, hyperline)
    2144415924382335131: "computers.network.professional",

    # TV digital / TDT (bbk, selenga, hyundai, perfeo, ritmix)
    2144415933341368710: "electronics.tv.digital",

    # Pilas y baterías (smartbuy, ultraflash, camelion, perfeo)
    2144415928148820220: "electronics.batteries",

    # Accesorios xiaomi / móviles (topon, xiaomi, pitatel, borofone)
    2144415940379411019: "electronics.mobile.accessories",

    # Ropa de trabajo / seguridad (sin marca dominante)
    2144415945681011417: "accessories.work_safety",

    # Electrodomésticos cocina medianos (polaris, redmond, unit)
    2144415923660914824: "appliances.kitchen.medium",

    # Redes / routers (huawei, tp-link, ubiquiti, mikrotik)
    2144415924189397141: "computers.network.routers",

    # Accesorios streaming / smart tv (palmexx, google, xiaomi, apple)
    2144415922125799520: "electronics.streaming",

    # Extensiones eléctricas (most, belkin, pilot, smartbuy)
    2144415928618582281: "electronics.power.extensions",

    # Herramientas manuales (ombra, makita, jonnesway, bort)
    2144415939431498289: "construction.tools.manual",

    # Baterías UPS (delta, csb, cyberpower, robiton)
    2144415938164818447: "electronics.power.batteries",

    # Accesorios auto (interpower, sky, swat, pioneer)
    2144415923384090754: "auto.accessories.generic",

    # Electrodomésticos cocina (scarlett, greys, luxell, kitfort)
    2144415929658769702: "appliances.kitchen.small",

    # Herramientas reparación móvil (kaisi, sirius, zhengte)
    2144415935732122060: "construction.tools.repair",

    # Máquinas de coser (merrylock, janome, comfort, necchi)
    2144415940119364164: "appliances.sewing",

    # Herramientas eléctricas 2 (makita, favourite, bort, metabo)
    2144415952459006878: "construction.tools.electric",

    # Accesorios cámara (canon, nikon, sony, rekam, fujifilm)
    2144415924784988326: "electronics.camera.accessories",

    # Cocinas / hornos (gefest, gorenje, bosch, lex)
    2144415933341368710: "appliances.kitchen.ovens",

    # Afeitadoras/depilación (philips, braun, panasonic, sinbo)
    2144415922905940086: "appliances.personal_care.shaving",

    # Drones (pilotage, skymoto, dji, syma)
    2144415941528650344: "electronics.drones",

    # Accesorios gaming (buro, steelseries, hama, logitech)
    2144415932368290154: "computers.gaming.accessories",

    # Refrigeración pasiva (zalman, titan, gembird, aerocool)
    2144415927947493623: "computers.components.cooling",

    # Pequeños electrodomésticos cocina 2 (redmond, kitfort, willmark)
    2144415923795132555: "appliances.kitchen.small",

    # Accesorios varios (sirius, samsung, asus, xiaomi, sony)
    2144415943516750494: "electronics.accessories.generic",

    # Cargadores auto (acv, palmexx)
    2144415921203052617: "auto.accessories.chargers",

    # Audio HiFi (pioneer, sony, yamaha, denon)
    2144415926252994761: "electronics.audio.hifi",

    # Accesorios almacenamiento 2 (palmexx, espada, orient, exegate)
    2144415927913939190: "computers.storage.accessories",

    # Herramientas eléctricas 3 (bosch, makita, jet)
    2144415951511094148: "construction.tools.electric",

    # Iluminación LED (camelion, smartbuy, ecola, nanoleaf, gauss)
    2144415929163841816: "electronics.lighting",

    # Accesorios auto 2 (acv, ural, selenga, bosch)
    2144415938416476694: "auto.accessories.generic",

    # Memorias flash 2 (sandisk, kingston, transcend, a-data)
    2144415921085612102: "computers.storage.flash",

    # Accesorios varios auto (acv, digma, rexant, swat)
    2144415939045622310: "auto.accessories.generic",

    # Pocketbook / e-readers (pocketbook, cameronsino, sony)
    2144415929440665888: "electronics.ereader",

    # Electrodomésticos varios (polaris, beurer, hyundai, scarlett)
    2144415927192518882: "appliances.small.generic",

    # Cables varios (telecom, cablexpert, hama, vcom)
    2144415939305669165: "computers.accessories.cables",

    # Herramientas eléctricas 4 (bosch, metabo, makita, bort)
    2144415926596927698: "construction.tools.electric",

    # Iluminación decorativa (twinkly, neon-night)
    2144415959924867186: "electronics.lighting.decorative",

    # Accesorios laptop (alas, topon, lenovo, hp)
    2144415935388189122: "computers.laptop.accessories",

    # Detectores / comprobadores (berkut, fubag, cobra, inspector)
    2144415929230950682: "construction.tools.detectors",

    # Herramientas eléctricas 5 (bosch, p.i.t, metabo, hammer)
    2144415952693887909: "construction.tools.electric",

    # Hubs USB / accesorios (baseus, orico, orient, hama)
    2144415926101999813: "computers.accessories.hubs",

    # Baterías cámara (pitatel, acmepower, ismartdigi, canon)
    2144415924894040233: "electronics.camera.batteries",

    # Carcasas móvil (samsung, spigen, coteetci, lyambda)
    2144415937552450047: "electronics.mobile.cases",

    # Audio portátil 2 (ritmix, perfeo, fiio, sony, digma)
    2144415921303715915: "electronics.audio.portable",

    # Fundas tablet/laptop (procase, exegate, supermicro, dell)
    2144415929012846868: "computers.laptop.cases",

    # Accesorios móvil 2 (ritmix, inspector, blast, multitronics)
    2144415921857364057: "electronics.mobile.accessories",

    # Termos y cantimploras (thermos, stanley, biostal, tiger)
    2144415945806840541: "accessories.outdoor.thermos",

    # Accesorios tablet (pitatel, digma, palmexx, buro)
    2144415927251239140: "electronics.tablet.accessories",

    # Radios (megajet, optim, midland, yaesu)
    2144415931328102737: "electronics.radio",

    # DVD / Blu-ray (verbatim)
    2144415929734267176: "electronics.storage.optical",

    # Radios amateur (baofeng, motorola, alan, yaesu)
    2144415926328492235: "electronics.radio.amateur",

    # Herramientas jardín 2 (bort, makita, bosch, metabo)
    2144415933660135822: "construction.tools.garden",

    # Tensiómetros / salud (b.well, endever, and, gmini)
    2144415937971880457: "appliances.health.bloodpressure",

    # Bolígrafos técnicos (rotring)
    2144415932762554741: "stationery.technical",

    # Accesorios hama (hama)
    2144415929776210217: "electronics.accessories.generic",

    # Reproductores mp3 (ritmix, digma, espada)
    2144415921739923542: "electronics.audio.mp3",

    # Routers / switches (huawei, zyxel, zte, mikrotik)
    2144415921983193180: "computers.network.routers",

    # Ollas / cocina (kelli, tefal, lara, scovo)
    2144415945555182293: "appliances.kitchen.cookware",

    # Organizadores oficina (kw-trio, deli, silwerhof)
    2144415930816397635: "stationery.office",

    # Tensiómetros (b.well, microlife)
    2144415933416866184: "appliances.health.bloodpressure",

    # Herramientas corte (fit, hualei, bosch, stayer)
    2144415959320887393: "construction.tools.cutting",

    # Fundas laptop (ritmix, digma, palmexx)
    2144415935421743555: "computers.laptop.cases",

    # Multímetros (kaisi, peakmeter, ekf, smartbuy, fluke)
    2144415972340007387: "construction.tools.measurement",

    # Accesorios cámara 2 (sony, palmexx, transcend, digma, rekam)
    2144415952635167651: "electronics.camera.accessories",

    # Utensilios cocina (kelli, goldenberg, sinbo, mallony)
    2144415929163841816: "appliances.kitchen.utensils",

    # Domótica / smart home (smartbuy, koogeek, fibaro, digma, ajax)
    2144415935195251132: "electronics.smarthome",

    # Cámaras de vigilancia (hikvision, tantos, dahua, rubetek)
    2144415944397554358: "electronics.security.cameras",

    # Cámaras drones (dji, pgytech)
    2144415954170282959: "electronics.camera.drone",

    # Cámaras de vigilancia 2 (hikvision, hiwatch, xiaomi, ezviz)
    2144415928366924034: "electronics.security.cameras",

    # Accesorios auto 3 (acv, intro, aura, avis)
    2144415939121119784: "auto.accessories.generic",

    # Mates y termos (mallony, teco, zepter, tefal)
    2144415928727634188: "appliances.kitchen.thermos",

    # Cargadores móvil (hoco, samsung, smartbuy, ldnio)
    2144415921463099471: "electronics.mobile.chargers",

    # Destructoras / encuadernadoras (rexel, fellowes, cactus, kobra)
    2144415928878629136: "stationery.shredders",

    # Trituradores / destructoras 2 (fellowes, gladwork, cactus, kobra)
    2144415933232316803: "stationery.shredders",

    # Smart home / ajax (ajax)
    2176606883765289435: "electronics.smarthome",
    2176606883798843868: "electronics.smarthome",

    # Acondicionadores aire (ballu, neoclima, hyundai)
    2144415941721588333: "appliances.climate.aircon",

    # Accesorios móvil 3 (baseus, deppa, remax, wk)
    2144415928106877179: "electronics.mobile.accessories",

    # Aspiradoras robot (redmond, bosch, karcher, iboto)
    2144415960939888782: "appliances.environment.vacuum",

    # Cafeteras / espresso (delonghi, melitta, philips)
    2144415929541329187: "appliances.kitchen.coffee",

    # Accesorios cámara 3 (nikon, canon)
    2144415924717879460: "electronics.camera.accessories",

    # Accesorios bolsas (coocazoo, hama, piquadro)
    2144415927813275891: "accessories.bags",

    # Accesorios auto 4 (cameronsino, pitatel)
    2144415943961346730: "auto.accessories.chargers",

    # Teléfonos de escritorio (panasonic, termit, fanvil, grandstream)
    2144415924617216161: "electronics.telephone.desk",

    # Jardín motorizado (huter, patriot, hyundai, husqvarna)
    2144415939397943856: "construction.tools.garden",

    # Cocinas vitrocerámica (gefest, cw, energy)
    2144415956049330180: "appliances.kitchen.stoves",

    # Navajas multifunción (leatherman, victorinox, gerber)
    2144415945890726624: "accessories.tools.multifunction",

    # Baterías externas (camelion, smartbuy, ecola)
    2144415929088344342: "electronics.mobile.powerbank",

    # Máquinas de escribir / impresoras etiquetas (zebra, cello)
    2144415932628337009: "stationery.printers",

    # Herramientas corte 2 (metabo, makita, bosch, felisatti)
    2144415952635167651: "construction.tools.cutting",

    # Cepillos dentales (philips, oral-b, panasonic)
    2144415925959393474: "appliances.personal_care.dental",

    # Estuches / fundas cámara (canon, nikon, hama, bresser)
    2144415926143942854: "electronics.camera.accessories",

    # Audio pro (ion, audio-technica, pioneer, sony)
    2144415942568837766: "electronics.audio.professional",

    # Herramientas manuales 2 (kraft, ombra, stayer)
    2144415941620925034: "construction.tools.manual",

    # Accesorios Microsoft / gaming (microsoft, cameronsino, pitatel)
    2144415928400478467: "electronics.gaming.accessories",

    # Herramientas eléctricas 6 (metabo, hammer, makita, bosch)
    2144415943877460648: "construction.tools.electric",

    # Herramientas corte 3 (bosch, metabo)
    2144415958926622806: "construction.tools.cutting",

    # Cargadoras / arrancadores (fubag, berkut, hyundai)
    2144415944758264511: "auto.accessories.chargers",

    # Switches / KVM (aten, d-link)
    2144415975443792438: "computers.network.switches",

    # Jardinería (gardena, raco, patriot, bosch)
    2144415943315423896: "construction.tools.garden",

    # Herramientas jardín 3 (huter, patriot)
    2144415943088931474: "construction.tools.garden",

    # Reproductores DVD (bbk, sony, hyundai, lg)
    2144415925128921263: "electronics.video.dvd",

    # Termómetros / salud 2 (beurer, sanitas)
    2144415957886435384: "appliances.health.thermometer",

    # Accesorios móvil carga (topon, xiaomi, pitatel, borofone)
    2144415940379411019: "electronics.mobile.accessories",

    # Herramientas soldadura 2 (fubag, metabo)
    2144415958322643012: "construction.tools.soldering",

    # Herramientas manuales jardín (stayer, kraftool)
    2144415938928181795: "construction.tools.garden",

    # Navajas (victorinox)
    2144415938257093137: "accessories.tools.multifunction",

    # Herramientas varias (bosch, makita, gardena)
    2150220533491302472: "construction.tools.electric",

    # Termostatos (beurer, centek, sanitas, rowenta)
    2144415938257093137: "appliances.health.generic",

    # Accesorios móvil xiaomi (xiaomi, rubetek)
    2144415978799235731: "electronics.mobile.accessories",

    # Accesorios fotografía (dji, pgytech)
    2144415954136728526: "electronics.camera.drone",

    # Proyectores (cactus, digma, palmexx, gaoke)
    2144415943416087195: "electronics.projectors",

    # Herramientas jardín 4 (makita)
    2265220528810460159: "construction.tools.garden",

    # Welding / soldadura (fubag, stayer, foxweld)
    2144415950991000437: "construction.tools.welding",

    # Termos outdoor (thermos, stanley, biostal)
    2144415945806840541: "accessories.outdoor.thermos",

    # Accesorios auto 5 (acv, ural)
    2144415938416476694: "auto.accessories.generic",

    # Generadores (fubag)
    2144415950546404201: "construction.tools.generators",

    # Herramientas abrasivas (bosch, makita, lugaabrasiv)
    2144415959287332960: "construction.tools.abrasive",

    # Electrodomésticos salud (b.well, philips)
    2144415942669501065: "appliances.health.generic",

    # Accesorios bolígrafos (rotring, zebra)
    2144415932594782576: "stationery.technical",

    # Alarmas auto (starline)
    2144415938525528601: "auto.security",

    # Aspiradoras jardín (karcher, bosch, huter)
    2144415952492561311: "construction.tools.garden",

    # Filtros agua (brita, bwt, karcher)
    2144415927846830324: "appliances.water",

    # Destructoras 3 (rexel, fellowes, cactus, kobra)
    2144415928878629136: "stationery.shredders",

    # Accesorios bolso (piquadro)
    2144415934314447266: "accessories.bags",

    # Motores / herramientas pesadas (p.i.t., bort, makita)
    2144415952660333476: "construction.tools.electric",

    # Cortacésped (gardena, raco, patriot, bosch)
    2144415943315423896: "construction.tools.garden",

    # Impresión 3D (cactus, funtastique, myriwell)
    2144415944900870851: "electronics.3dprinting",

    # Radios bluetooth (blast)
    2144415975578010170: "electronics.audio.portable",

    # Cámaras seguridad (hikvision, dahua, activision)
    2144415975150191149: "electronics.security.cameras",

    # Accesorios pc (oklick)
    2223384012628427485: "computers.accessories.generic",

    # Herramientas abrasivas 2 (bosch, metabo)
    2144415958851125332: "construction.tools.abrasive",

    # Accesorios tablet 2 (xiaomi, rubetek)
    2144415978799235731: "electronics.tablet.accessories",

    # Accesorios robots (ubtech)
    2144415941822251632: "electronics.smarthome",

    # Lavadoras a presión (karcher, phantom)
    2144415959480270949: "appliances.environment.pressure_washer",

    # Walkie-talkies (termit, goip, yeastar)
    2144415957341175849: "electronics.radio.professional",

    # Termómetros herramientas (fluke, bosch, stayer)
    2144415937586004480: "construction.tools.measurement",

    # Detectores metales (berkut, fubag, cobra)
    2144415929230950682: "construction.tools.detectors",

    # Herramientas corte 4 (p.i.t., bort, makita)
    2144415931084833099: "construction.tools.cutting",

    # Redes switches (d-link, gigalink, tp-link)
    2144415930178863411: "computers.network.switches",

    # Aspiradoras vapor (rowenta, philips, polaris, braun)
    2144415977289286248: "appliances.environment.vacuum",

    # Herramientas eléctricas 7 (metabo, jet, patriot, bosch)
    2144415943290258071: "construction.tools.electric",

    # Herramientas abrasivas 3 (bosch, makita, dremel)
    2144415928652136714: "construction.tools.abrasive",

    # Paraguas / accesorios outdoor (camping, fiesta, irit)
    2144415944070398637: "accessories.outdoor.generic",

    # Repelentes (thermacell, outdoor, irit)
    2144415936503874018: "accessories.outdoor.repellents",

    # Cafeteras 2 (kitfort)
    2144415941755142766: "appliances.kitchen.coffee",

    # Planchas ropa (scarlett, endever, mallony)
    2144415959639654506: "appliances.ironing",

    # Herramientas eléctricas 8 (bort, bosch, patriot)
    2144415937250460151: "construction.tools.electric",

    # Herramientas corte 5 (stayer)
    2144415972465836511: "construction.tools.cutting",

    # Accesorios cámara 4 (fujifilm, polaroid)
    2144415928190763261: "electronics.camera.accessories",

    # Teléfonos inalámbricos (motorola, trendnet)
    2144415936638091750: "electronics.telephone.wireless",

    # Limpieza industrial (cleanfix)
    2144415975578010170: "appliances.cleaning.industrial",

    # Soldaduras (fubag, p.i.t.)
    2144415950546404201: "construction.tools.welding",

    # Herramientas medición 2 (stayer, kraftool)
    2144415938928181795: "construction.tools.measurement",

    # Herramientas jardín 5 (bosch, makita, gardena)
    2150220533491302472: "construction.tools.garden",

    # Accesorios Wahl (wahl)
    2144415976836301404: "appliances.personal_care.hair",

    # Archivadores (leitz)
    2144415932368290154: "stationery.office",

    # Impresoras etiquetas 2 (zebra, cello)
    2144415932628337009: "stationery.printers",

    # Accesorios camping (acecamp)
    2144415978992173720: "accessories.outdoor.camping",

    # Cables profesionales (coroplast)
    2144415975619953211: "computers.accessories.cables",

    # Herramientas fubag (fubag)
    2144415932561228143: "construction.tools.welding",

    # Cámaras hikvision 2 (tantos, dahua, activision, falcon)
    2144415975150191149: "electronics.security.cameras",

    # Gillette / afeitado (gillette)
    2144415946117219047: "appliances.personal_care.shaving",

    # Accesorios viaje (coocazoo)
    2144415932368290154: "accessories.travel",

    # Electrónica náutica (dors, pro, moniron)
    2144415929574883620: "electronics.marine",

    # Herramientas neumáticas (fubag, stayer, foxweld)
    2144415932561228143: "construction.tools.pneumatic",

    # Cajas herramientas (ombra, hikoki)
    2151554989560955734: "construction.tools.storage",

    # Inyectores (iek)
    2151554989393183573: "construction.tools.electric",

    # Motosierras (huter, patriot)
    2144415943088931474: "construction.tools.chainsaws",

    # Cables (ekf)
    2150220529112449031: "computers.accessories.cables",

    # Accesorios fotografía 2 (stayer, kraftool)
    2144415938928181795: "construction.tools.manual",

    # Herramientas makita (makita)
    2265220528810460159: "construction.tools.electric",

    # Accesorios hiper (hiper)
    2144415975175356974: "computers.accessories.generic",

    # Accesorios digma (digma)
    2166248785527701775: "electronics.accessories.generic",

    # Accesorios stayer (stayer)
    2144415974177112595: "construction.tools.accessories",

    # Matavientos / encendedores (grinda)
    2144415943315423896: "accessories.outdoor.generic",

    # Accesorios dexx (dexx)
    2151554994141135726: "electronics.accessories.generic",

    # Robiton (robiton)
    2151554992396305254: "electronics.batteries",

    # Accesorios iek (iek)
    2144415971484369346: "construction.tools.electric",

    # Cámaras de vigilancia 3 (hikvision)
    2144415941822251632: "electronics.security.cameras",

    # Radios 2 (motorola)
    2144415936638091750: "electronics.radio",

    # Dendy consolas (dendy)
    2144415981139657427: "electronics.gaming",

    # Accesorios bosch (bosch)
    2144415978572743308: "construction.tools.electric",

    # Termómetros beurer (beurer)
    2144415926415873928: "appliances.health.thermometer",

    # Aspiradoras centek (centek, gorenje)
    2144415970494513573: "appliances.environment.vacuum",

    # UPS / energía (wolta, iek)
    2144415959639654506: "electronics.power.ups",

    # Camping (greys, tefal)
    2144415966954520896: "accessories.outdoor.camping",

    # Accesorios bolso 2 (piquadro)
    2144415967969542492: "accessories.bags",

    # Dispensadores agua (thetford)
    2144415942258459260: "appliances.water",

    # Rómbiton pilas (robiton)
    2144415946150773480: "electronics.batteries",

    # Herramientas eléctricas 9 (bosch, metabo, bort)
    2144415932594782576: "construction.tools.electric",

    # Nanoleaf (nanoleaf)
    2144415972465836511: "electronics.lighting",

    # Nokia accesorios (nokia)
    2144415965478125844: "electronics.mobile.accessories",

    # Patria herramientas (patriot)
    2144415958809182291: "construction.tools.electric",

    # Microlife (microlife)
    2144415973891899915: "appliances.health.bloodpressure",

    # Braun (braun)
    2144415965553623318: "appliances.personal_care.shaving",

    # Beurer salud (beurer)
    2144415949783040852: "appliances.health.generic",

    # Nika (nika)
    2144415947333567245: "electronics.accessories.generic",

    # B.well (b.well)
    2144415933903405461: "appliances.health.generic",

    # Metabo accesorios (metabo)
    2144415972398727645: "construction.tools.accessories",

    # Kraftool (kraftool)
    2144415971232711099: "construction.tools.manual",

    # Smartbuy (smartbuy)
    2144415924860485800: "electronics.accessories.generic",

    # Accesorios janome (janome)
    2144415973313085946: "appliances.sewing",

    # Tefal (tefal)
    2144415946117219047: "appliances.kitchen.cookware",

    # Phantom (phantom)
    2144415975502512696: "auto.accessories.generic",

    # Accesorios makita (makita)
    2144415978497245834: "construction.tools.accessories",

    # Accesorios ombra (ombra)
    2144415942996656784: "construction.tools.accessories",

    # Accesorios bosch 2 (bosch)
    2144415971920576974: "construction.tools.accessories",

    # Iek (iek)
    2151554993050616681: "construction.tools.electric",

    # Patriot herramientas (patriot)
    2144415967256510792: "construction.tools.electric",

    # Accesorios stayer 2 (stayer)
    2144415937929937416: "construction.tools.accessories",

    # Bellavita (bellavita)
    2144415964840591616: "appliances.personal_care",

    # Delta (delta)
    2144415968338641254: "electronics.power.batteries",

    # Karcher (karcher)
    2144415973774459400: "appliances.environment.pressure_washer",

    # Endever (endever)
    2144415966988075329: "appliances.small.generic",

    # Mallony (mallony)
    2144415966862246205: "appliances.kitchen.cookware",

    # Coroplast (coroplast)
    2144415975619953211: "computers.accessories.cables",

    # Kraftool 2 (kraftool)
    2144415971585032645: "construction.tools.manual",

    # Accesorios fotografía 3 (stayer)
    2144415975653507644: "construction.tools.accessories",

    # Raco (raco)
    2144415973313085946: "accessories.outdoor.generic",

    # Teco (teco)
    2144415959287332960: "construction.tools.generic",

    # Park (park)
    2144415968145703265: "auto.accessories.generic",

    # Accesorios grinda (grinda)
    2144415970863612336: "construction.tools.garden",

    # Accesorios universales
    2144415970628731305: "electronics.accessories.generic",
    2144415970494513573: "electronics.accessories.generic",

    # Ekf (ekf)
    2144415972340007387: "construction.tools.measurement",

    # Accesorios varios
    2144415966652530999: "electronics.accessories.generic",
    2144415963372585172: "construction.tools.accessories",
    2144415963347419347: "construction.tools.accessories",
    2144415962097516719: "appliances.health.generic",
    2144415962449838264: "appliances.personal_care",
    2144415965436182803: "construction.tools.generic",
    2144415965369073937: "construction.tools.generic",
    2144415964421161204: "appliances.small.generic",
    2144415964479881462: "accessories.outdoor.generic",
    2144415957919989817: "accessories.outdoor.generic",
    2144415957634777137: "construction.tools.electric",
    2144415957508948013: "construction.tools.electric",
    2144415958280699971: "electronics.accessories.generic",
    2144415959178281053: "construction.tools.electric",
    2144415959010508888: "construction.tools.abrasive",
    2144415959111172187: "construction.tools.accessories",
    2144415959044063321: "construction.tools.electric",
    2144415959362830434: "construction.tools.accessories",
    2144415959220224094: "electronics.security.generic",
    2144415959253778527: "electronics.security.generic",
    2144415959740317805: "electronics.lighting",
    2144415960906334349: "appliances.kitchen.small",
    2144415960939888782: "appliances.environment.vacuum",
    2144415960000364660: "computers.accessories.generic",
    2144415961218695907: "electronics.accessories.generic",
}

In [9]:
# APLICAR EL MAPA
nulos_antes = df["category_code"].isna().sum()

df["category_code"] = df.apply(
    lambda row: mapa_categorias.get(row["category_id"], row["category_code"])
    if pd.isna(row["category_code"])
    else row["category_code"],
    axis=1,
)

nulos_despues = df["category_code"].isna().sum()
recuperados = nulos_antes - nulos_despues

print(f"\nNulos antes:     {nulos_antes:,}")
print(f"Recuperados:     {recuperados:,} ({recuperados/nulos_antes*100:.1f}%)")
print(f"Aún sin inferir: {nulos_despues:,} ({nulos_despues/nulos_antes*100:.1f}%)")



Nulos antes:     236,172
Recuperados:     212,633 (90.0%)
Aún sin inferir: 23,539 (10.0%)


In [10]:
# Los que no se pudieron inferir → "accessories.generic"
df["category_code"] = df["category_code"].fillna("accessories.generic")
df["category_inferred"] = df.apply(
    lambda row: True if row["category_id"] in mapa_categorias else False,
    axis=1,
)

print(f"\nTop 15 categorías después de inferencia:")
print(df["category_code"].value_counts().head(15))


Top 15 categorías después de inferencia:
category_code
computers.components.videocards     116712
electronics.telephone                84343
computers.peripherals.printer        43219
stationery.cartrige                  38719
electronics.audio.acoustic           26764
computers.components.motherboard     26600
computers.notebook                   25025
computers.components.cpu             24768
accessories.generic                  23539
electronics.video.tv                 21391
electronics.telephone.accessory      20771
electronics.tablet                   19376
auto.accessories.player              17501
computers.components.cooling         13903
computers.accessories.cables         11058
Name: count, dtype: int64


## 2.4 INFERENCIA DE BRAND — productos sin marca
asignar "generic." + ultima_parte_de_la_categoria. 
Ejemplo: electronics.telephone → generic.telephone 
computers.components.videocards → generic.videocard

In [12]:
nulos_brand_antes = df["brand"].isna().sum()
print(f"\nFilas sin brand: {nulos_brand_antes:,}")


Filas sin brand: 212,326


In [13]:
# PASO 1: intentar recuperar via diccionario product_id → brand
dict_brand = (
    df[df["brand"].notna()]
    .groupby("product_id")["brand"]
    .first()
    .to_dict()
)

df["brand"] = df.apply(
    lambda row: dict_brand.get(row["product_id"], row["brand"])
    if pd.isna(row["brand"])
    else row["brand"],
    axis=1,
)

recuperados_dict = nulos_brand_antes - df["brand"].isna().sum()
print(f"Recuperados vía diccionario: {recuperados_dict:,}")

Recuperados vía diccionario: 0


In [14]:
# PASO 2: aplicar inferencia para filas aún sin brand
brand_inferida_previa = (
    df["brand_inferred"].fillna(False)
    if "brand_inferred" in df.columns
    else pd.Series(False, index=df.index)
)


def inferir_brand(row):
    category_code = row.get("category_code")
    if pd.notna(category_code):
        category_code = str(category_code).strip()
        if category_code:
            return f"generic.{category_code.split('.')[-1]}"

    return "generic.unknown"


mascara_inferir_brand = df["brand"].isna() | brand_inferida_previa

df.loc[mascara_inferir_brand, "brand"] = df.loc[mascara_inferir_brand].apply(
    inferir_brand,
    axis=1,
)

# Agregar columna que indica si brand fue inferida
df["brand_inferred"] = mascara_inferir_brand

In [15]:
# VERIFICACIÓN
print("\n" + "=" * 55)
print("RESULTADO FINAL")
print("=" * 55)

nulos_final = df["brand"].isna().sum()
inferidas = df["brand_inferred"].sum()

print(f"\nNulos originales:      {nulos_brand_antes:,}")
print(f"Recuperados diccionario: {recuperados_dict:,}")
print(f"Inferidas por categoría: {inferidas:,}")
print(f"Nulos restantes:         {nulos_final:,}")

print(f"\nTop 15 marcas después de inferencia:")
print(df["brand"].value_counts().head(15))

print(f"\nMarcas genéricas más frecuentes:")
print(df[df["brand_inferred"]]["brand"].value_counts().head(15))



RESULTADO FINAL

Nulos originales:      212,326
Recuperados diccionario: 0
Inferidas por categoría: 212,326
Nulos restantes:         0

Top 15 marcas después de inferencia:
brand
generic.telephone     37327
asus                  27703
gigabyte              27673
msi                   24876
palit                 24801
samsung               23198
amd                   20107
canon                 18437
generic.videocards    17158
generic.generic       16349
panasonic             11986
pioneer               11467
sirius                11404
hp                    11186
generic.cartrige      10883
Name: count, dtype: int64

Marcas genéricas más frecuentes:
brand
generic.telephone      37327
generic.videocards     17158
generic.generic        16349
generic.cartrige       10883
generic.accessory      10689
generic.printer         9707
generic.cables          9599
generic.notebook        6761
generic.soldering       6345
generic.accessories     5699
generic.tablet          4968
generic.shelvin

In [16]:
df

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,category_inferred,brand_inferred
0,2020-09-24 11:57:06 UTC,view,1996170,2144415922528452715,electronics.telephone,generic.telephone,31.90,1515915625519388267,LJuJVLEjPT,False,True
1,2020-09-24 11:57:26 UTC,view,139905,2144415926932472027,computers.components.cooler,zalman,17.16,1515915625519380411,tdicluNnRY,False,False
2,2020-09-24 11:57:27 UTC,view,215454,2144415927158964449,computers.accessories.cables,generic.cables,9.81,1515915625513238515,4TMArHtXQy,True,True
3,2020-09-24 11:57:33 UTC,view,635807,2144415923107266682,computers.peripherals.printer,pantum,113.81,1515915625519014356,aGFYrNgC08,False,False
4,2020-09-24 11:57:36 UTC,view,3658723,2144415921169498184,electronics.telephone.accessory,cameronsino,15.87,1515915625510743344,aa4mmk0kwQ,True,False
...,...,...,...,...,...,...,...,...,...,...,...
885124,2021-02-28 23:55:01 UTC,view,953226,2144415927553229037,construction.tools.garden,generic.garden,219.94,1515915625611023730,FRLqIttxKU,True,True
885125,2021-02-28 23:58:05 UTC,view,1715907,2144415927049912542,electronics.video.tv,starwind,80.03,1515915625611024014,g6WqPf50Ma,False,False
885126,2021-02-28 23:58:09 UTC,view,4170534,2144415939364389423,electronics.clocks,amazfit,64.92,1515915625611024020,xNIJBqZdkd,False,False
885127,2021-02-28 23:58:14 UTC,view,888273,2144415921932861531,electronics.telephone,generic.telephone,10.16,1515915625611024030,9pCbKMIcSx,False,True


In [17]:
df[df["brand"] == "unknown"]

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,category_inferred,brand_inferred


In [18]:
df['category_code'].nunique(dropna=True)

216

In [19]:
# Valores únicos de la columna 'category_code'
df['category_code'].nunique()
print("Número de valores únicos en 'category_code':", df['category_code'].nunique())
# Valores únicos de la columna 'category_code'; 20 primeros
df['category_code'].unique()[:20]

Número de valores únicos en 'category_code': 216


array(['electronics.telephone', 'computers.components.cooler',
       'computers.accessories.cables', 'computers.peripherals.printer',
       'electronics.telephone.accessory', 'construction.tools.saw',
       'computers.desktop', 'computers.network.router',
       'construction.tools.manual', 'electronics.audio.portable',
       'electronics.accessories.generic', 'electronics.video.tv',
       'auto.accessories.player', 'computers.components.motherboard',
       'electronics.ereader', 'electronics.camera.video',
       'accessories.generic', 'computers.peripherals.keyboard',
       'computers.network.professional', 'construction.tools.garden'],
      dtype=object)

In [20]:
# Identificación de estructura jerárquica en 'category_code'
# Contar la cantidad de niveles jerárquicos
df['category_code'].str.count('\.').value_counts().sort_index()

# Mostrar ejemplos de cada nivel jerárquico
for i in range(4):  # Asumiendo un máximo de 4 niveles jerárquicos y sumatoria total por cada niveljerarquico.
    print(f"\nTotal de valores en nivel {i} (con {i} puntos):")
    print(df[df['category_code'].str.count('\.') == i].shape[0])
    print(f"Ejemplos de nivel {i} (con {i} puntos):")
    print(df[df['category_code'].str.count('\.') == i]['category_code'].unique()[:5])



Total de valores en nivel 0 (con 0 puntos):
0
Ejemplos de nivel 0 (con 0 puntos):
[]

Total de valores en nivel 1 (con 1 puntos):
250245
Ejemplos de nivel 1 (con 1 puntos):
['electronics.telephone' 'computers.desktop' 'electronics.ereader'
 'accessories.generic' 'appliances.personal_care']

Total de valores en nivel 2 (con 2 puntos):
634304
Ejemplos de nivel 2 (con 2 puntos):
['computers.components.cooler' 'computers.accessories.cables'
 'computers.peripherals.printer' 'electronics.telephone.accessory'
 'construction.tools.saw']

Total de valores en nivel 3 (con 3 puntos):
415
Ejemplos de nivel 3 (con 3 puntos):
['electronics.audio.music_tools.piano']


In [21]:
# Dividir la columna 'category_code' en niveles jerárquicos (nivel1, nivel2, nivel3, nivel4).
# Posteriormente, los registros que contienen nivel4 (casos poco frecuentes) se integran en nivel3
# para reducir la dispersión y evitar valores nulos innecesarios.
# Las celdas faltantes se representan como NaN.


# Dividir la columna
df[['nivel1', 'nivel2', 'nivel3', 'nivel4']] = df['category_code'].str.split('.', expand=True)

# Unir nivel3 + nivel4 SOLO cuando exista nivel4
df['nivel3'] = df['nivel3'].where(
    df['nivel4'].isnull(),
    df['nivel3'] + '.' + df['nivel4']
)

# Eliminar nivel4
df.drop(columns=['nivel4'], inplace=True)

df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,category_inferred,brand_inferred,nivel1,nivel2,nivel3
0,2020-09-24 11:57:06 UTC,view,1996170,2144415922528452715,electronics.telephone,generic.telephone,31.90,1515915625519388267,LJuJVLEjPT,False,True,electronics,telephone,None
1,2020-09-24 11:57:26 UTC,view,139905,2144415926932472027,computers.components.cooler,zalman,17.16,1515915625519380411,tdicluNnRY,False,False,computers,components,cooler
2,2020-09-24 11:57:27 UTC,view,215454,2144415927158964449,computers.accessories.cables,generic.cables,9.81,1515915625513238515,4TMArHtXQy,True,True,computers,accessories,cables
3,2020-09-24 11:57:33 UTC,view,635807,2144415923107266682,computers.peripherals.printer,pantum,113.81,1515915625519014356,aGFYrNgC08,False,False,computers,peripherals,printer
4,2020-09-24 11:57:36 UTC,view,3658723,2144415921169498184,electronics.telephone.accessory,cameronsino,15.87,1515915625510743344,aa4mmk0kwQ,True,False,electronics,telephone,accessory


In [22]:
# validar la consistencia de la información entre los niveles jerárquicos y la columna original 'category_code'

''' “Se validó la consistencia de la transformación jerárquica mediante la reconstrucción de la variable original,
obteniendo una concordancia del 100%. Asimismo, se verificó la correcta integración del cuarto nivel en el tercer nivel,
alcanzando una correspondencia total entre los valores esperados y transformados.”
'''


# reconstrucción = Se toman los niveles, elimina nulos, los une en formato original y mide qué porcentaje coincide con la columna original.
df['reconstructed'] = df.apply(
    lambda row: '.'.join(
        [str(x) for x in [row['nivel1'], row['nivel2'], row['nivel3']] if pd.notnull(x)]
    ),
    axis=1
)

# limpiar posibles puntos dobles o finales
df['reconstructed'] = df['reconstructed'].str.strip('.').str.replace(r'\.+', '.', regex=True)

# comparación
print("Reconstrucción:", (df['category_code'] == df['reconstructed']).mean())

# validación de nivel3
df['expected_nivel3'] = df['category_code'].str.split('.', n=2).str[2]

match = (
    (df['nivel3'] == df['expected_nivel3']) |
    (df['nivel3'].isna() & df['expected_nivel3'].isna())
).mean()

print("Match nivel3:", match)

Reconstrucción: 1.0
Match nivel3: 1.0


In [23]:
# Eliminar columnas de validación
df.drop(columns=['reconstructed', 'expected_nivel3'], inplace=True)

In [24]:
# validar casos específicos de la categoría 'music_tools.piano' para verificar la correcta integración del nivel4 en nivel3.
df[df['category_code'].str.contains('music_tools.piano')][
    ['category_code', 'nivel3']
].head()

,category_code,nivel3
1374,electronics.audio.music_tools.piano,music_tools.piano
8121,electronics.audio.music_tools.piano,music_tools.piano
9344,electronics.audio.music_tools.piano,music_tools.piano
10602,electronics.audio.music_tools.piano,music_tools.piano
25169,electronics.audio.music_tools.piano,music_tools.piano


In [25]:
# muestra de la información final del dataframe, cantidad de filas, nulos por columna y unicidad de cada nivel jerárquico.

print("Filas:", len(df))
print("\nNulos por columna:")
print(df[['nivel1','nivel2','nivel3']].isnull().sum())

print("\nUnicidad:")
print("nivel1:", df['nivel1'].nunique())
print("nivel2:", df['nivel2'].nunique())
print("nivel3:", df['nivel3'].nunique())

Filas: 884964

Nulos por columna:
nivel1         0
nivel2         0
nivel3    250245
dtype: int64

Unicidad:
nivel1: 14
nivel2: 75
nivel3: 140


In [26]:
# muestra de los espacios y caracteres raros en los niveles jerárquicos para asegurar la limpieza de los datos.

# espacios
print("Espacios en nivel1:", df['nivel1'].str.contains(' ', na=False).sum())
print("Espacios en nivel2:", df['nivel2'].str.contains(' ', na=False).sum())
print("Espacios en nivel3:", df['nivel3'].str.contains(' ', na=False).sum())

# caracteres raros
import re
pattern = r'[^a-zA-Z0-9_\.]'
print("Chars raros nivel2:", df['nivel2'].str.contains(pattern, regex=True, na=False).sum())
print("Chars raros nivel3:", df['nivel3'].str.contains(pattern, regex=True, na=False).sum())

Espacios en nivel1: 0
Espacios en nivel2: 0
Espacios en nivel3: 0
Chars raros nivel2: 0
Chars raros nivel3: 0


In [27]:
# Ambieguedad semantica entre niveles jerárquicos 

# Obteniendo valores únicos de cada nivel jerárquico
set_n1 = set(df['nivel1'].dropna().unique())
set_n2 = set(df['nivel2'].dropna().unique())
set_n3 = set(df['nivel3'].dropna().unique())

print("Intersección entre nivel1 y nivel2:", set_n1.intersection(set_n2))
print("Intersección entre nivel1 y nivel3:", set_n1.intersection(set_n3))
print("Intersección entre nivel2 y nivel3:", set_n2.intersection(set_n3))

Intersección entre nivel1 y nivel2: {'accessories'}
Intersección entre nivel1 y nivel3: {'accessories'}
Intersección entre nivel2 y nivel3: {'accessories', 'storage', 'batteries', 'video', 'small', 'generic', 'camera', 'kitchen'}


In [28]:
# categorias presentes en los tres niveles jerarquicos
all_levels = set_n1.intersection(set_n2).intersection(set_n3)
print("En los 3 niveles:", all_levels)
print("Categorías presentes en los tres niveles jerárquicos:", all_levels)

En los 3 niveles: {'accessories'}
Categorías presentes en los tres niveles jerárquicos: {'accessories'}


Se identificaron categorías con ambigüedad semántica al aparecer en múltiples niveles jerárquicos. 
    Se evaluó su consistencia respecto al nivel superior, permitiendo diferenciar entre categorías 
    coherentes e inconsistentes para aplicar normalización selectiva.

In [ ]:

    
# verificando que las categorias de nivel 2 y nivel 3 pertenecen al mismo nivel 1 o distinto nivel.
cats_n2_n3 = set(df['nivel2'].dropna().unique()).intersection(
    set(df['nivel3'].dropna().unique())
)

problematic = []

for cat in cats_n2_n3:
    n1_n2 = set(df[df['nivel2'] == cat]['nivel1'])
    n1_n3 = set(df[df['nivel3'] == cat]['nivel1'])
    
    print(f"\n📌 {cat}")
    print("nivel2 → nivel1:", n1_n2)
    print("nivel3 → nivel1:", n1_n3)
    
    if n1_n2 == n1_n3:
        print("✅ Consistente")
    else:
        print("⚠️ Inconsistente (revisar)")


📌 accessories
nivel2 → nivel1: {'electronics', 'auto', 'computers'}
nivel3 → nivel1: {'electronics', 'construction', 'computers'}
⚠️ Inconsistente (revisar)

📌 storage
nivel2 → nivel1: {'electronics', 'computers'}
nivel3 → nivel1: {'construction'}
⚠️ Inconsistente (revisar)

📌 batteries
nivel2 → nivel1: {'electronics'}
nivel3 → nivel1: {'electronics', 'construction'}
⚠️ Inconsistente (revisar)

📌 video
nivel2 → nivel1: {'electronics'}
nivel3 → nivel1: {'electronics'}
✅ Consistente

📌 small
nivel2 → nivel1: {'appliances'}
nivel3 → nivel1: {'appliances'}
✅ Consistente

📌 generic
nivel2 → nivel1: {'accessories'}
nivel3 → nivel1: {'accessories', 'construction', 'auto', 'electronics', 'appliances', 'computers'}
⚠️ Inconsistente (revisar)

📌 camera
nivel2 → nivel1: {'electronics'}
nivel3 → nivel1: {'computers'}
⚠️ Inconsistente (revisar)

📌 kitchen
nivel2 → nivel1: {'furniture', 'appliances'}
nivel3 → nivel1: {'appliances'}
⚠️ Inconsistente (revisar)


In [30]:
sorted(df['nivel1'].unique())

['accessories',
 'apparel',
 'appliances',
 'auto',
 'computers',
 'construction',
 'country_yard',
 'electronics',
 'furniture',
 'jewelry',
 'kids',
 'medicine',
 'sport',
 'stationery']

Se identificaron categorías con ambigüedad semántica al presentarse en múltiples niveles jerárquicos. 
Se aplicó una normalización contextual incorporando niveles superiores, mejorando la consistencia y 
calidad de las variables para el sistema de recomendación.

In [ ]:
# lista de categorias problematicas
problematic = ['storage', 'accessories', 'kitchen', 'generic', 'batteries', 'camera']

map_n1 = {
    'accessories': 'acc',
    'apparel': 'app',
    'appliances': 'appl',
    'auto': 'auto',
    'computers': 'comp',
    'construction': 'const',
    'country_yard': 'cyard',
    'electronics': 'elec',
    'furniture': 'furn',
    'jewelry': 'jew',
    'kids': 'kids',
    'medicine': 'med',
    'sport': 'sport',
    'stationery': 'stat'
}


In [32]:
# normalización contextual para categorías problemáticas en nivel2, integrando el nivel1 correspondiente para mejorar la consistencia semántica.

# crear copia base
df['nivel2_norm'] = df['nivel2']

# máscara
mask_n2 = df['nivel2'].isin(problematic)

# aplicar transformación
df.loc[mask_n2, 'nivel2_norm'] = (
    df.loc[mask_n2, 'nivel1'].map(map_n1) + '_' + df.loc[mask_n2, 'nivel2']
)

In [33]:
# normalización contextual para categorías problemáticas en nivel3, donde se usará nivel2_norm.

#crear copia base
df['nivel3_norm'] = df['nivel3']

# máscara
mask_n3 = df['nivel3'].isin(problematic)

#aplicar transformación
df.loc[mask_n3, 'nivel3_norm'] = (
    df.loc[mask_n3, 'nivel2_norm'] + '_' + df.loc[mask_n3, 'nivel3']
)

In [34]:
# validación de la normalización contextual aplicada a nivel2 y nivel3, verificando que las categorías problemáticas 
# se hayan transformado correctamente con el prefijo del nivel superior correspondiente.

df[df['nivel3'].isin(problematic)][
    ['nivel1', 'nivel2', 'nivel2_norm', 'nivel3', 'nivel3_norm']
].head(10)

,nivel1,nivel2,nivel2_norm,nivel3,nivel3_norm
12,electronics,accessories,elec_accessories,generic,elec_accessories_generic
32,electronics,accessories,elec_accessories,generic,elec_accessories_generic
41,electronics,accessories,elec_accessories,generic,elec_accessories_generic
70,computers,peripherals,peripherals,camera,peripherals_camera
71,computers,peripherals,peripherals,camera,peripherals_camera
82,computers,peripherals,peripherals,camera,peripherals_camera
84,construction,tools,tools,batteries,tools_batteries
86,auto,accessories,auto_accessories,generic,auto_accessories_generic
103,construction,tools,tools,batteries,tools_batteries
114,construction,tools,tools,batteries,tools_batteries


In [35]:
# conteo total de generic y porcentaje respecto al total de registros para evaluar su impacto en el dataset.

total_generic = (df['nivel3'] == 'generic').sum()
percentage_generic = (total_generic / len(df)) * 100
print("Total registros con 'generic':", total_generic)
print("Porcentaje de registros con 'generic': {:.2f}%".format(percentage_generic))

Total registros con 'generic': 12820
Porcentaje de registros con 'generic': 1.45%


In [36]:
# Mostrar todas las filas duplicadas (considerando todas las columnas)
duplicados = df[df.duplicated(keep=False)]

print(f"Total de filas duplicadas: {len(duplicados)}")
display(duplicados)

Total de filas duplicadas: 1278


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,category_inferred,brand_inferred,nivel1,nivel2,nivel3,nivel2_norm,nivel3_norm
511,2020-09-24 13:51:07 UTC,view,387956,2144415922427789416,computers.components.videocards,asus,104.21,1515915625519429853,PZu2caZ5EN,False,False,computers,components,videocards,components,videocards
512,2020-09-24 13:51:07 UTC,view,387956,2144415922427789416,computers.components.videocards,asus,104.21,1515915625519429853,PZu2caZ5EN,False,False,computers,components,videocards,components,videocards
974,2020-09-24 15:48:55 UTC,view,874667,2144415922738167921,computers.components.cdrw,asus,23.48,1515915625519457150,8wvs0vbHtv,False,False,computers,components,cdrw,components,cdrw
975,2020-09-24 15:48:55 UTC,view,874667,2144415922738167921,computers.components.cdrw,asus,23.48,1515915625519457150,8wvs0vbHtv,False,False,computers,components,cdrw,components,cdrw
4827,2020-09-25 13:15:09 UTC,view,453469,2144415924222951574,auto.accessories.parktronic,generic.parktronic,69.84,1515915625519725870,9ofICyh8Eo,False,True,auto,accessories,parktronic,auto_accessories,parktronic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
878751,2021-02-27 17:56:05 UTC,view,1571204,2144415924491387038,computers.components.motherboard,asus,146.40,1515915625610505518,EUqy2lyCvY,False,False,computers,components,motherboard,components,motherboard
879544,2021-02-27 20:54:20 UTC,view,1027953,2144415923837075596,electronics.audio.acoustic,jbl,332.87,1515915625529755153,3mD3HIQ017,False,False,electronics,audio,acoustic,audio,acoustic
879545,2021-02-27 20:54:20 UTC,view,1027953,2144415923837075596,electronics.audio.acoustic,jbl,332.87,1515915625529755153,3mD3HIQ017,False,False,electronics,audio,acoustic,audio,acoustic
882715,2021-02-28 14:18:02 UTC,view,4078916,2144415922427789416,computers.components.videocards,sapphire,415.54,1515915625610828170,21hX1rWtum,False,False,computers,components,videocards,components,videocards


In [37]:
# Eliminar duplicados y dejar solo 1 fila por cada grupo igual
df_sin_duplicados = df.drop_duplicates(keep='first')

print(f"Filas originales: {len(df)}")
print(f"Filas sin duplicados: {len(df_sin_duplicados)}")
print(f"Duplicados eliminados: {len(df) - len(df_sin_duplicados)}")

display(df_sin_duplicados)

Filas originales: 884964
Filas sin duplicados: 884312
Duplicados eliminados: 652


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,category_inferred,brand_inferred,nivel1,nivel2,nivel3,nivel2_norm,nivel3_norm
0,2020-09-24 11:57:06 UTC,view,1996170,2144415922528452715,electronics.telephone,generic.telephone,31.90,1515915625519388267,LJuJVLEjPT,False,True,electronics,telephone,None,telephone,None
1,2020-09-24 11:57:26 UTC,view,139905,2144415926932472027,computers.components.cooler,zalman,17.16,1515915625519380411,tdicluNnRY,False,False,computers,components,cooler,components,cooler
2,2020-09-24 11:57:27 UTC,view,215454,2144415927158964449,computers.accessories.cables,generic.cables,9.81,1515915625513238515,4TMArHtXQy,True,True,computers,accessories,cables,comp_accessories,cables
3,2020-09-24 11:57:33 UTC,view,635807,2144415923107266682,computers.peripherals.printer,pantum,113.81,1515915625519014356,aGFYrNgC08,False,False,computers,peripherals,printer,peripherals,printer
4,2020-09-24 11:57:36 UTC,view,3658723,2144415921169498184,electronics.telephone.accessory,cameronsino,15.87,1515915625510743344,aa4mmk0kwQ,True,False,electronics,telephone,accessory,telephone,accessory
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
885124,2021-02-28 23:55:01 UTC,view,953226,2144415927553229037,construction.tools.garden,generic.garden,219.94,1515915625611023730,FRLqIttxKU,True,True,construction,tools,garden,tools,garden
885125,2021-02-28 23:58:05 UTC,view,1715907,2144415927049912542,electronics.video.tv,starwind,80.03,1515915625611024014,g6WqPf50Ma,False,False,electronics,video,tv,video,tv
885126,2021-02-28 23:58:09 UTC,view,4170534,2144415939364389423,electronics.clocks,amazfit,64.92,1515915625611024020,xNIJBqZdkd,False,False,electronics,clocks,None,clocks,None
885127,2021-02-28 23:58:14 UTC,view,888273,2144415921932861531,electronics.telephone,generic.telephone,10.16,1515915625611024030,9pCbKMIcSx,False,True,electronics,telephone,None,telephone,None


In [38]:
# Corroboracion: verificar que no existan duplicados
duplicados_post = df_sin_duplicados[df_sin_duplicados.duplicated(keep=False)]
print(f"Total de duplicados despues de limpiar: {len(duplicados_post)}")

Total de duplicados despues de limpiar: 0


In [39]:
#Se hace la imputación del resto de valores faltantes en el nivel 3 de categorías (nivel13_norm), con base en el nombre del nivel 2 y añadiendo _generic
df['nivel3_norm'] = df['nivel3_norm'].fillna(df['nivel2_norm'] + "_generic")

In [40]:
#e verifica la presencia de valores faltantes luego de la imputación
df['nivel3_norm'].isna().sum()

np.int64(0)

In [ ]:
# Exportar CSV limpio sin duplicados
os.makedirs("../data/final", exist_ok=True)
df_sin_duplicados.to_csv("../data/final/events_final.csv", index=False)
print("Archivo exportado como events_final.csv")

Archivo exportado como events_final.csv
